# YouTube Transcript Reverse-Engineer (Updated with Gradio UI)

This small Week1 challenge demonstrates use of multiple LLM calls to craft a new script based on the reverse-engineered video's  style, tone, structure, etc...

<i>P.S. The prompts are unrefined, but gpt-5.4-mini works the best out of the cheap models</i>

In [1]:
import os
import dotenv
from openai import OpenAI
from youtube_transcript_api import YouTubeTranscriptApi
import gradio as gr

In [2]:
CHATGPT_MODEL = "gpt-5.4-mini"
MODEL_LLAMA = 'llama3.2'
OLLAMA_BASE_URL = 'http://localhost:11434/v1'
GROQ_BASE_URL = 'https://api.groq.com/openai/v1'

In [3]:
dotenv.load_dotenv()
openai = OpenAI()
ollama = OpenAI(base_url=OLLAMA_BASE_URL)
groq = OpenAI(api_key= os.getenv("GROQ_API_KEY"), base_url=GROQ_BASE_URL)
api_key = os.getenv("OPENAI_API_KEY")
ytt_api = YouTubeTranscriptApi()

if not api_key:
  raise ValueError("OPENAI_API_KEY environment variable not found. Please set it in your .env file.")
else:
  print("OPENAI_API_KEY successfully loaded from .env file.")

OPENAI_API_KEY successfully loaded from .env file.


In [4]:
reverse_engineer_system_prompt = """
You are a prompt engineering expert that is able to reverse engineer prompts based on the text that is provided to you.
I am going to provide you with YouTube transcripts that I want you to reverse engineer. 
Be as specific as possible on providing a prompting suggestion based on the tone, style, syntax, language, and any other factors you consider relevant. 
I would like to use the prompts in the future to replicate the style of the provided texts.
Your prompts are effective if, when entered into ChatGPT in a normal context, it would provide the script.
Please make sure to avoid including details that work only in the context of the provided video, and instead focus on the style, tone, structure, and syntax of the script.
Also, make sure to write the prompt itself without any additional commentary or information. Only provide the prompt.
Reply in markdown format.
"""

reverse_engineer_user_prefix = """Here is the YouTube transcript from a video that I want you to reverse engineer a prompt for:
"""

In [5]:
def format_video_transcript(unpolished_transcript):
  transcript = ""
  for snippet in unpolished_transcript:
    transcript += snippet.text + " "
  return transcript.strip()

In [6]:
def get_youtube_transcript(video_url):
  unpolished_transcript = ytt_api.fetch(video_url.split("v=")[-1])
  transcript = format_video_transcript(unpolished_transcript.snippets)
  return transcript

In [7]:
def get_reverse_engineered_prompt(new_title, title, url):
  transcript = get_youtube_transcript(url)
  user_message = f"Reverse engineer the script for this new title: {new_title} \n {reverse_engineer_user_prefix} {title} \n {transcript}"
  response = openai.chat.completions.create(
    model=CHATGPT_MODEL,
    messages=[{"role": "system", "content": reverse_engineer_system_prompt}, {"role": "user", "content": user_message}]
  )
  return response.choices[0].message.content

In [8]:
system_prompt = """You're a professional script writer for YouTube videos. 
You create engaging, informative, and entertaining scripts that captivate audiences. 
You write according to a provided style, tone, structure and syntax.
You have a knack for storytelling and can break down intricate subjects into digestible pieces while maintaining the viewer's interest throughout the video.
Replay in markdown format."
"""

In [9]:
def generate_new_script(new_title, title, url):
  reverse_engineered_user_prompt = get_reverse_engineered_prompt(new_title, title, url)
  stream = openai.chat.completions.create(
    model=CHATGPT_MODEL,
    messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": reverse_engineered_user_prompt}],
    stream=True
  )
  script = ""
  for chunk in stream:
    script += chunk.choices[0].delta.content or ""
    yield script

In [10]:
def generate_new_script_from_ollama(new_title, title, url):
    reverse_engineered_user_prompt = get_reverse_engineered_prompt(new_title, title, url)
    stream = groq.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": reverse_engineered_user_prompt}],
        stream=True
    )
    script = ""
    for chunk in stream:
        script += chunk.choices[0].delta.content or ""
        yield script

In [11]:
def get_model(new_title, title, url, model="GPT"):
    if model == "GPT":
        result = generate_new_script(new_title, title, url)
    elif model == "LLaMA":
        result = generate_new_script_from_ollama(new_title, title, url)
    else:
        print("Invalid model name: " + model + ". Please choose 'GPT' or 'LLaMA'.")
  
    yield from result

In [12]:
new_title_input = gr.Textbox(label="New Title", info="Enter a title you want the script to be about")
title_input = gr.Textbox(label="Title", info="Enter a title of the YouTube video to be reverse engineered")
url_input = gr.Textbox(label="URL", info="Enter a URL of the video")
output = gr.Markdown(label="Reverse-Engineered Script")
dropdown = gr.Dropdown(label="Model", choices=["GPT", "LLaMA"], value="GPT")
view = gr.Interface(
  fn=get_model, 
  title="YouTube Script Reverse-Engineer",
  inputs=[new_title_input, title_input, url_input, dropdown],
  outputs=[output],
  flagging_mode="never"
)

view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7910
* To create a public link, set `share=True` in `launch()`.


In [ ]:
generate_new_script(
  new_title="How Hackers Pirate Every Single Netflix Movie!", 
  title="How Hackers Crack Every Single Game!", 
  url="https://www.youtube.com/watch?v=GEX5jIs2BHs"
)

```markdown
Write a YouTube script in the same style as a fast-paced, curiosity-driven explainer video about how hackers pirate every single Netflix movie.

Use a conversational, dramatic, slightly sarcastic tone with simple language, vivid analogies, and strong pacing. Start with a specific historical or technical hook that makes the platform seem “uncrackable,” then immediately undercut it with how quickly pirates found a way around it. Explain the process in plain English as if talking to a general audience, using comparisons like bouncers, VIP passes, locked doors, fake keys, or invisible security guards.

Structure the script like this:
- Open with a bold claim and a surprising contradiction
- Explain the original protection system in simple terms
- Walk through how pirates/hackers bypassed it step by step
- Introduce the modern anti-piracy system and why it is harder
- Describe the tools, methods, and workflow attackers use
- Emphasize how the process evolved over time
- End with

**[OPENING HOOK]**

When Netflix first launched, people treated it like a fortress.  
Hollywood studios saw it and basically said, “Sure, stream our movies… what’s the worst that could happen?”

Well.  
The answer was: **piracy happened. Fast.**

Because the truth is, no streaming platform is truly “uncrackable.”  
It’s not a magic castle. It’s more like a really fancy building with a thousand doors, a few security guards, and one very determined person wearing a fake mustache.

So how did hackers and pirates get around Netflix-style protection systems?  
And why is it still such a constant battle today?

Let’s break it down.

---

## **[1. THE ORIGINAL PROTECTION SYSTEM]**

At the beginning, the idea was simple.

Netflix didn’t just send you a movie file and hope for the best.  
That would be like handing out free copies of a movie on a USB stick and saying, “Please don’t share this.”

Instead, they used **digital rights management**, or **DRM**.  
That’s the fancy term for “security locks on digital content.”

Think of DRM like a nightclub.

The movie is inside.  
But you don’t get in just because you know the address.  
You need a VIP pass. A bouncer checks your ID. The music only plays if you’re on the list.

On Netflix, your device gets a special permission token.  
That token says, “Yes, this person is allowed to watch this movie.”

And the video itself is usually **encrypted**.  
That means the file is scrambled into unreadable nonsense unless you have the right key.

So the system looked strong.  
Locked doors. Security guards. VIP passes. End of story, right?

But then… pirates showed up.

---

## **[2. HOW PIRATES GOT AROUND IT]**

Here’s the catch.

If you want to watch a movie, your device has to **unscramble** it at some point.  
Because otherwise, all you’d have is a pile of digital gibberish.

So the question becomes:  
Where does the movie turn from scrambled code into something you can actually see?

And that’s where things get interesting.

Pirates didn’t always try to break the whole fortress from the outside.  
Sometimes, they just found a window when the guards were distracted.

Early on, one of the biggest tricks was **capturing the video after decryption**.  
In plain English: they waited until the movie was unlocked on the user’s device, then copied the clean version before it reached the screen.

It’s like sneaking into a bank after the vault opens and photographing the money while the teller is counting it.  
You didn’t crack the vault.  
You just caught the treasure at the wrong moment.

Some attackers used **screen capture** methods.  
That means recording what was being displayed on the screen.  
Not glamorous. Not elegant. But it works.

Others went deeper.  
They looked for **keys stored in memory**.  
Memory is the short-term workspace of a computer.  
If the key exists there, even briefly, a determined attacker may be able to find it.

So the movie wasn’t really “stolen” from Netflix’s servers in the classic sense.  
It was often captured from the playback process itself.

That’s a crucial difference.

Netflix could protect the door.  
But if someone got inside the room with a camera, that was another problem.

---

## **[3. STEP BY STEP: THE BYPASS METHOD]**

So how does that work in practice?

At a high level, the workflow looked something like this:

**Step 1: Get access to a valid playback environment.**  
That could mean a real device, a browser session, or a modified app.

**Step 2: Wait for the content to decrypt.**  
The content has to be decrypted locally so it can play.  
No decryption, no viewing.

**Step 3: Intercept the output.**  
Instead of attacking the encrypted file, attackers target the stream after it’s been unlocked.

**Step 4: Repackage or record the clean video.**  
At that point, they can save it, duplicate it, and spread it.

That’s the ugly genius of it.  
Not breaking the lock.  
Just copying the thing after the lock has already opened.

And yes, this got better over time.

Because pirates aren’t just random people with laptops and bad intentions.  
Some of them are very patient.  
Very technical.  
And very good at finding the tiny cracks in a system designed to be “good enough.”

---

## **[4. FAST FORWARD TO TODAY: MODERN ANTIVI-PIRACY]**

Fast forward to today, and Netflix-style protection is much more advanced.

Now we’re talking about systems like **Widevine**, **PlayReady**, and **FairPlay**.  
Those are DRM frameworks used by major platforms to control playback.

Basically, they’re different brands of invisible security guards.

And they don’t just check whether you have a pass.  
They also check **what kind of device** you’re using, whether it’s trusted, and whether the environment looks tampered with.

Modern systems can use **hardware-based encryption** too.  
That means some of the keys are protected inside secure chips, not just sitting around in software where they’re easier to poke at.

That’s a lot harder to mess with.

Why?  
Because if the key lives inside a secure hardware zone, it’s like the movie is stored in a safe that only opens when the right guard, the right room, and the right fingerprint all line up.

So now pirates have a much tougher job.

They can’t just lean on simple recording tricks and call it a day.  
They have to deal with better detection, stronger encryption, and more layers of defense.

But… and this is a big but… no system is perfect.

---

## **[5. THE TOOLS, METHODS, AND WORKFLOW]**

So what do attackers actually use?

Not one magic weapon.  
Usually a whole toolbox.

They may use:

- **Modified playback software**
- **Debugging tools**, which are programs used to inspect how software works
- **Automation scripts**, which repeat tasks over and over
- **Memory analysis tools**, which look inside a program while it runs
- **Capture devices** or recording setups
- **Custom patches** to bypass checks or disable protections

In simple terms, it’s a bit like trying to rob a museum where every hallway has motion sensors, every painting has a lock, and the cameras never sleep.

So what do the thieves do?

They study the routine.  
They look for the blind spots.  
They wait for the one employee who leaves a side door open.  
Then they move fast.

A lot of modern piracy is about **workflow**, not just hacking.

One person figures out how to defeat the protection on a specific device or app version.  
Another person records or extracts the content.  
Someone else cleans it up, labels it, and distributes it.

It’s not always one mastermind in a hoodie typing in the dark.  
Sometimes it’s an entire chain of people using specialized tools, each doing one small part.

And that’s why the whole thing keeps evolving.

---

## **[6. HOW THE PROCESS EVOLVED OVER TIME]**

At first, piracy was crude.

Low quality.  
Camera recordings.  
Audio that sounded like it was trapped in a tin can.  
Movies with people coughing in the background.

Then it got cleaner.

Better capture methods.  
Better software.  
Better understanding of how streaming devices handle encrypted content.

And now, the battle is more like an arms race.

Netflix and other platforms keep tightening security.  
Pirates keep searching for new cracks.  
Security gets patched.  
New methods appear.  
The cycle repeats.

It’s not a one-time defeat.  
It’s a permanent tug-of-war.

And the funny thing is, the real battleground isn’t always the movie file itself.  
It’s the path the movie takes from server to screen.

That path is where the drama lives.  
That’s where the lock is, the key is, and the sneaky hand is waiting to grab it.

---

## **[7. THE MORAL TAKEAWAY]**

So yes, piracy has always found ways around streaming security.  
Sometimes through technical flaws.  
Sometimes through loopholes.  
Sometimes through sheer persistence.

But that doesn’t mean the system is pointless.  
It means digital protection is always playing catch-up with people trying to break it.

And here’s the bigger picture:  
Movies, shows, music, games—these are made by real people.  
Writers. Editors. Artists. Developers. Crews.  
When content gets stolen, the damage isn’t abstract.  
It affects the people who made it.

Still, the tech side is fascinating, right?  
A constant battle between locks and fake keys.  
Between invisible guards and people trying to slip past them.

So what do you think?

Are streaming platforms winning this fight, or are pirates always going to find the next loophole?  
Drop your opinion in the comments.